# Decoding & Sampling Strategies

**Course:** [Natural Language Processing](https://ml-viz-ruby.vercel.app/courses/nlp/05-decoding-and-sampling)

**The idea in one sentence.** A language model only ever outputs a *probability
distribution* over the next token — **decoding** is the separate, weight-free
step that turns that distribution into an actual token, and the strategy you pick
trades **determinism** against **diversity**.

That decoupling is the key insight: greedy, temperature, top-$k$, top-$p$, and
beam search are all pure functions of the logits. You can swap them at inference
time with no retraining, which is why the same model can be a precise code
assistant (low temperature) or a brainstorming partner (high temperature).

We implement **every** common decoder from scratch in NumPy, **validate** our
softmax against SciPy and confirm sampling frequencies converge to the true
distribution, then map out the deployment gotchas.

No API keys, no network, no PyTorch. Just NumPy + matplotlib on a small
hand-crafted next-token distribution.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

BRAND = '#6366f1'
TEAL = '#2dd4bf'
ORANGE = '#f97316'
ROSE = '#fb7185'
MUTED = '#475569'

## 1. A toy logits distribution

We'll use a hand-crafted 8-token vocabulary for the prompt `"The cat sat on the ___"`, identical to the one driving the on-site `<SamplingViz />` component. Each decoder below operates on these raw logits.

In [ ]:
TOKENS = ['mat', 'floor', 'sofa', 'roof', 'table', 'bed', 'grass', 'moon']
LOGITS = np.array([3.2, 2.6, 2.1, 1.0, 1.7, 1.3, 0.4, -0.5])

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

probs = softmax(LOGITS)
print('Token        Logit   Probability')
print('-' * 36)
for tok, lg, p in zip(TOKENS, LOGITS, probs):
    print(f'{tok:<10}  {lg:+5.2f}    {p:.3f}')
print(f'\nTotal probability: {probs.sum():.6f}')

## 2. The four sampling primitives

Every modern decoder is some composition of these four building blocks. All four are pure functions of the next-token distribution — they live entirely *outside* the trained weights.

In [ ]:
def greedy(logits):
    '''Argmax of the logits. Equivalent to beam search with k=1 and to sampling at T=0.'''
    return int(np.argmax(logits))

def temperature(logits, T):
    '''Rescale logits by T before softmax. T->0 = greedy; T->infinity = uniform.'''
    return softmax(logits / max(T, 1e-9))

def top_k(probs, k):
    '''Keep only the k highest-probability tokens; renormalise.'''
    if k >= len(probs):
        return probs.copy()
    cutoff = np.partition(probs, -k)[-k]
    out = np.where(probs >= cutoff, probs, 0.0)
    return out / out.sum()

def top_p(probs, p):
    '''Smallest set whose cumulative probability >= p, renormalised. Adaptive truncation.'''
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    # Smallest k such that cum[k-1] >= p
    k = int(np.searchsorted(cum, p) + 1)
    k = min(k, len(probs))
    keep = np.zeros_like(probs)
    keep[order[:k]] = probs[order[:k]]
    return keep / keep.sum()

# Quick sanity checks
print('greedy(LOGITS)    ->', TOKENS[greedy(LOGITS)])
print('softmax(LOGITS)   ->', np.round(temperature(LOGITS, 1.0), 3))
print('top_k(p, k=3)     ->', np.round(top_k(probs, 3), 3))
print('top_p(p, p=0.9)   ->', np.round(top_p(probs, 0.9), 3))

Read off the toy example:

- **Greedy** always picks `mat` (highest logit).
- **Top-$k$ at $k = 3$** keeps `mat`, `floor`, `sofa` and zeros the rest.
- **Top-$p$ at $p = 0.9$** keeps whichever prefix of the sorted probabilities first sums to $\ge 0.9$ — the size of that set adapts to the shape of the distribution.

### Validate: softmax matches SciPy, and sampling converges to the distribution

Two checks. First, our hand-rolled `softmax` must equal `scipy.special.softmax`
(the numerically-stable reference). Second — the whole premise of *sampling* —
if we draw many tokens under temperature $T=1$, the **empirical frequencies**
must converge to the model's probabilities (law of large numbers). If they
didn't, "sampling from the distribution" would be a lie.

In [ ]:
from scipy.special import softmax as sp_softmax

assert np.allclose(softmax(LOGITS), sp_softmax(LOGITS)), 'our softmax must match scipy'
print('softmax vs scipy: max abs diff =', np.abs(softmax(LOGITS) - sp_softmax(LOGITS)).max())

rng_v = np.random.default_rng(0)
N = 200_000
draws = rng_v.choice(len(TOKENS), size=N, p=probs)
emp = np.bincount(draws, minlength=len(TOKENS)) / N
print(f'\n{"token":<8}{"true p":>9}{"empirical":>11}')
for t, pt, pe in zip(TOKENS, probs, emp):
    print(f'{t:<8}{pt:9.3f}{pe:11.3f}')
assert np.max(np.abs(emp - probs)) < 0.01, 'empirical frequencies must converge to the true distribution'
print('\n✅ softmax is correct and sampling reproduces the distribution (max dev < 0.01)')

## 3. Temperature sweep: $T = 0.3, 1.0, 1.5$

The classic "peakier vs flatter" picture. Low temperature concentrates mass on the top token (near-greedy); high temperature flattens the distribution toward uniform.

In [ ]:
Ts = [0.3, 1.0, 1.5]
colors = [BRAND, TEAL, ORANGE]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, T, c in zip(axes, Ts, colors):
    p = temperature(LOGITS, T)
    ax.bar(TOKENS, p, color=c, edgecolor='#0f1117')
    ax.set_title(f'T = {T}   (entropy = {-(p*np.log2(p+1e-12)).sum():.2f} bits)', fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Probability')
plt.suptitle('Temperature reshapes the distribution', fontsize=12, y=1.04)
plt.tight_layout()
plt.show()

At $T = 0.3$ the top token's probability is huge and entropy is near zero — almost greedy. At $T = 1.5$ the distribution is much flatter; entropy approaches the uniform-limit value $\log_2 8 = 3$ bits.

## 4. Top-$k$ vs top-$p$ side by side

Both truncate, but top-$k$ is fixed-count and top-$p$ is fixed-mass. We compare $k = 3$ vs $k = 6$ and $p = 0.5$ vs $p = 0.95$ on the same base distribution.

In [ ]:
configs = [
    ('top_k, k=3',  top_k(probs, 3),  BRAND),
    ('top_k, k=6',  top_k(probs, 6),  TEAL),
    ('top_p, p=0.5', top_p(probs, 0.5), ORANGE),
    ('top_p, p=0.95', top_p(probs, 0.95), ROSE),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
for ax, (title, p, c) in zip(axes, configs):
    kept = p > 0
    bar_colors = [c if k else MUTED for k in kept]
    ax.bar(TOKENS, p, color=bar_colors, edgecolor='#0f1117')
    ax.set_title(f'{title}  ({int(kept.sum())} kept)', fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Probability (renormalised)')
plt.suptitle('Top-k vs top-p: fixed count vs fixed mass', fontsize=12, y=1.04)
plt.tight_layout()
plt.show()

Notice the *adaptivity* of top-$p$. At $p = 0.5$ only the top two tokens make the cut (their mass alone already exceeds half the distribution); at $p = 0.95$ six tokens are kept. Top-$k$, by contrast, keeps the same count regardless of how confident the model happens to be at this step. That difference is why nucleus sampling is the open-ended-generation default in modern LLMs.

## 5. Beam search on a synthetic 4-token continuation

Beam search keeps the top-$k$ partial sequences scored by **cumulative log-probability**. We'll build a tiny synthetic next-token distribution that depends on the current prefix and run beam search out four tokens with $k = 3$.

To keep the demo readable: the vocabulary is `['a', 'b', 'c']`, and the next-token distribution at every step is a small deterministic function of the last token. The point is mechanism, not realism.

In [ ]:
VOC = ['a', 'b', 'c']

# next_logits[last_token] -> logits over VOC
NEXT_LOGITS = {
    None: np.array([2.0, 1.0, 0.5]),   # initial step
    'a':  np.array([0.3, 1.8, 1.2]),
    'b':  np.array([1.5, 0.2, 1.4]),
    'c':  np.array([1.0, 1.6, 0.4]),
}

def step_logprobs(last):
    return np.log(softmax(NEXT_LOGITS[last]))

def beam_search(k, T):
    '''Length-T beam search with beam width k, returns list of (sequence, cum_logprob).'''
    beams = [(['<s>'], 0.0)]  # (tokens, cumulative log-prob)
    for _ in range(T):
        candidates = []
        for seq, lp in beams:
            last = seq[-1] if seq[-1] != '<s>' else None
            lps = step_logprobs(last)
            for i, w in enumerate(VOC):
                candidates.append((seq + [w], lp + float(lps[i])))
        # Keep top-k by cumulative log-prob
        candidates.sort(key=lambda c: c[1], reverse=True)
        beams = candidates[:k]
    return beams

results = beam_search(k=3, T=4)
print('Top-3 beams after 4 tokens:')
print('-' * 50)
for seq, lp in results:
    text = ' '.join(seq[1:])  # drop <s>
    print(f'  {text:<20}  cum log-prob = {lp:+.3f}   (p = {np.exp(lp):.4f})')

Compare to greedy: a one-step-argmax decoder would emit `a` at step 1 (highest logit) and then have to live with whatever distribution `a` induces at step 2. Beam search keeps several first-step candidates alive long enough to see whether a less-locally-likely opener leads to a globally better continuation — that's exactly the *Canberra / Sydney* recovery from the lesson.

## 6. Monte-Carlo diversity: sampling 5 000 sequences

To make "diversity vs determinism" concrete, we'll sample 5 000 one-token completions from the toy distribution under several (temperature, top-$p$) recipes and plot the empirical entropy of each one. Higher entropy = more diverse generations.

In [ ]:
def sample_once(rng, logits, T, p):
    distribution = temperature(logits, T)
    distribution = top_p(distribution, p)
    return int(rng.choice(len(distribution), p=distribution))

recipes = [
    ('greedy (T=0)',           0.001, 1.0,  BRAND),
    ('chat (T=0.7, p=0.9)',    0.7,   0.9,  TEAL),
    ('creative (T=1.0, p=0.95)', 1.0, 0.95, ORANGE),
    ('untruncated T=1.5',      1.5,   1.0,  ROSE),
]

N = 5000
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, len(recipes), figsize=(15, 3.5), sharey=True)
print(f"{'Recipe':<28}{'unique':>8}{'top token':>14}{'entropy (bits)':>18}")
print('-' * 70)
for ax, (label, T, p, c) in zip(axes, recipes):
    samples = [sample_once(rng, LOGITS, T, p) for _ in range(N)]
    counts = np.bincount(samples, minlength=len(TOKENS))
    emp_p = counts / counts.sum()
    H = -(emp_p[emp_p > 0] * np.log2(emp_p[emp_p > 0])).sum()
    top_idx = int(np.argmax(counts))
    print(f'{label:<28}{(counts>0).sum():>8d}{TOKENS[top_idx]:>14}{H:>18.3f}')
    ax.bar(TOKENS, counts / N, color=c, edgecolor='#0f1117')
    ax.set_title(f'{label}\nH = {H:.2f} bits', fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Empirical frequency')
plt.suptitle('Diversity of 5 000 one-token samples across recipes', fontsize=12, y=1.06)
plt.tight_layout()
plt.show()

Read the table column by column. **Greedy** has entropy zero (one token, every time). The **chat recipe** $(T = 0.7, p = 0.9)$ picks from a handful of tokens with a clear mode — that's exactly the trade-off you want for a helpful assistant. **Creative** spreads further across the nucleus. **Untruncated $T = 1.5$** has the highest entropy but also samples genuinely implausible tokens — gibberish in a real model. The picture is the deployment knob: pick the column whose entropy matches the task.

## Gotchas & tradeoffs

| Strategy | Failure mode | When to use |
|----------|--------------|-------------|
| **greedy / T=0** | repetition loops, bland text, "the the the" | deterministic tasks (code, extraction, math) |
| **high temperature, no truncation** | samples genuinely implausible tokens → gibberish | almost never alone — always pair with top-$k$/top-$p$ |
| **top-$k$ (fixed count)** | too small = repetitive; ignores how peaked the step is | simple, cheap; $k \approx 40$–$50$ |
| **top-$p$ (fixed mass)** | adapts to confidence — the modern default | open-ended generation; $p \approx 0.9$–$0.95$ |
| **beam search** | high-probability ≠ interesting; degenerates to repetition on open-ended text | closed-ended tasks (translation, summarisation) |

Demo: greedy decoding on a repetitive transition table falls into a **loop** —
the classic "neural text degeneration" that motivated nucleus sampling.

In [ ]:
# A transition table where the greedy choice cycles: a->b->a->b...
TRANS = {'a': np.array([0.1, 0.9]), 'b': np.array([0.9, 0.1])}   # over ['a','b']
VOC2 = ['a', 'b']
def greedy_gen(start, n):
    seq = [start]
    for _ in range(n):
        seq.append(VOC2[int(np.argmax(TRANS[seq[-1]]))])
    return seq
def sample_gen(start, n, rng, T=1.0):
    seq = [start]
    for _ in range(n):
        p = temperature(np.log(TRANS[seq[-1]] + 1e-12), T)
        seq.append(VOC2[int(rng.choice(2, p=p))])
    return seq

rng_g = np.random.default_rng(1)
g = ''.join(greedy_gen('a', 12))
s = ''.join(sample_gen('a', 12, rng_g))
print(f'greedy   : {g}   <- locked in a 2-cycle, zero new information')
print(f'sampled  : {s}   <- breaks the loop')
assert g == 'ababababababa', 'greedy should fall into the a/b loop'
print('\nGreedy maximises per-step probability but can trap the model in a repetition loop;')
print('stochastic decoding (with truncation) is what keeps open-ended generation alive.')

## ✏️ Your turn

### Exercise: implement `top_p_student(probs, p)`

Implement nucleus sampling from scratch. Given a non-negative probability vector summing to 1 and a target mass $p \in (0, 1]$, return a renormalised vector where exactly the smallest prefix of tokens (sorted by descending probability) whose cumulative mass first reaches $p$ is kept; the rest are zeroed.

This is the routine sitting inside every modern LLM inference server.

In [ ]:
def top_p_student(probs, p):
    '''
    Smallest descending-sorted set of indices whose cumulative probability
    is >= p. Zero everything else, renormalise the survivors so they sum to 1.

    Args:
        probs: 1-D numpy array of non-negative floats summing to ~1
        p: target cumulative mass, in (0, 1]

    Returns:
        1-D numpy array of the same shape, summing to 1, with only the
        nucleus tokens non-zero.
    '''
    # TODO(you): implement the four steps
    #   1. sort indices by probability, descending
    #   2. find the smallest k such that cumulative sum >= p
    #   3. zero out everything outside the top-k
    #   4. renormalise so the kept entries sum to 1
    pass

# Quick visual check on a hand-built distribution
demo = np.array([0.40, 0.25, 0.15, 0.10, 0.05, 0.03, 0.02])
print('demo  ->', np.round(top_p_student(demo, 0.9), 3) if top_p_student(demo, 0.9) is not None else 'not implemented')

In [ ]:
# Tests
demo = np.array([0.40, 0.25, 0.15, 0.10, 0.05, 0.03, 0.02])

# Case 1: p = 0.9 -> cumulative [0.40, 0.65, 0.80, 0.90, ...]
# Smallest set reaching >= 0.9 is the first 4 tokens.
out = top_p_student(demo, 0.9)
assert out is not None, 'Should return an array, not None'
out = np.asarray(out)
assert out.shape == demo.shape, f'Expected shape {demo.shape}, got {out.shape}'
assert abs(out.sum() - 1.0) < 1e-9, f'Result must sum to 1, got {out.sum()}'
kept = (out > 0).sum()
assert kept == 4, f'p=0.9 should keep exactly 4 tokens on this distribution, kept {kept}'
# The kept four must be the four highest-probability indices in demo
assert set(np.argsort(out)[-4:].tolist()) == {0, 1, 2, 3}

# Case 2: very small p should keep exactly one token (the argmax)
out1 = np.asarray(top_p_student(demo, 0.1))
assert (out1 > 0).sum() == 1, 'p < probs.max() should keep exactly one token'
assert int(np.argmax(out1)) == int(np.argmax(demo))

# Case 3: p = 1.0 should keep every non-zero token
out_full = np.asarray(top_p_student(demo, 1.0))
assert (out_full > 0).sum() == len(demo)
assert abs(out_full.sum() - 1.0) < 1e-9

# Case 4: peaky distribution -> peakier nucleus
peaky = np.array([0.95, 0.02, 0.01, 0.01, 0.005, 0.003, 0.002])
peaky = peaky / peaky.sum()
out_peaky = np.asarray(top_p_student(peaky, 0.9))
assert (out_peaky > 0).sum() == 1, 'On a peaky distribution at p=0.9, nucleus should be 1 token'

print('✅ Exercise passed')

<details>
<summary>💡 Show solution</summary>

```python
def top_p_student(probs, p):
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    # Smallest k such that cum[k-1] >= p
    k = int(np.searchsorted(cum, p) + 1)
    k = min(k, len(probs))
    out = np.zeros_like(probs)
    out[order[:k]] = probs[order[:k]]
    return out / out.sum()
```

Three subtle points:

1. **`np.searchsorted(cum, p)`** returns the leftmost index where `p` could be inserted to keep `cum` sorted. We need the smallest $k$ such that `cum[k-1] >= p`, so we add 1.
2. The `min(k, len(probs))` guard matters for the edge case $p = 1.0$ on a distribution that doesn't quite sum to 1 due to float error.
3. The renormalisation step is what makes top-$p$ a valid sampling distribution — without it the kept entries would sum to a value $\le 1$ but $\ne 1$, and `np.random.choice` would complain (or silently mis-sample).
</details>

## Key takeaways

- **Decoding is separate from the model.** The network emits a distribution;
  greedy / temperature / top-$k$ / top-$p$ / beam are weight-free functions of it,
  swappable at inference time.
- **Temperature reshapes; truncation clips.** $T$ sharpens ($<1$) or flattens
  ($>1$) the distribution; top-$k$ keeps a fixed *count*, top-$p$ a fixed *mass*
  (adaptive — the modern default).
- **Sampling really is sampling.** We verified empirical frequencies converge to
  the true probabilities, and that our softmax matches SciPy.
- **Greedy/beam maximise probability, not quality.** On open-ended text they fall
  into **repetition loops**; nucleus sampling exists to prevent that degeneration.
- **Match the knob to the task:** low temperature / greedy for code and
  extraction; $T\approx0.7,\,p\approx0.9$ for chat; higher for brainstorming.